# Day 2-01｜手動偵測框到鳥瞰圖投影
> Python 籃球運動資料分析課程  
> 本單元使用固定球員框，說明 bbox bottom-center 作為地面接觸點近似值的用途。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 用 `[x1, y1, x2, y2]` 表示一個球員偵測框。
- 計算偵測框底邊中心點作為 footpoint。
- 將 footpoint 投影到 BEV 並檢查結果是否合理。

## 完成產出
- 包含相機畫面 bbox、footpoint 與 BEV 投影點的對照圖。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 載入範例影像與 Homography。
2. 設定一個手動 bbox。
3. 調整 bbox 並觀察 BEV 點位變化。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
from src.cv_utils import (
    read_image_rgb,
    draw_boxes,
    draw_points,
    show_image,
    side_by_side,
    load_json,
    bottom_center,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points, render_bev_court

image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = render_bev_court(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json")
info = load_json(COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json")
H = compute_homography(info["camera_points"], info["bev_points"])

bbox = info["sample_player_bbox_xyxy"]
foot = bottom_center(bbox)
bev_foot = project_points([foot], H)[0]

print("bbox:", bbox)
print("bottom-center:", foot)
print("projected BEV:", bev_foot)

In [ ]:
frame_vis = draw_boxes(image, [bbox], ["player bbox"])
frame_vis = draw_points(frame_vis, [foot], ["bottom-center"])
bev_vis = draw_points(bev, [bev_foot], ["player"])
out = side_by_side(frame_vis, bev_vis)
show_image(out, "左：一個人框；右：投影到 BEV 的點")

save_path = COURSE_ROOT / "assets" / "results" / "d2_01_manual_box_to_bev.png"
save_image_rgb(save_path, out)
print("saved:", save_path)

課堂操作：修改 `bbox` 四個數值，記錄 footpoint 與 BEV 投影點的變化。